# Calibration slope and calibration-in-the-large

Reviewers asked for calibration slope and calibration-in-the-large (CITL) alongside the Brier score and calibration error already reported.

The baseline model is cell D of the factorial: patient-grouped 5-fold CV, no SMOTE, class weighting, identical hyperparameters. This notebook regenerates its out-of-fold predictions and computes the calibration measures with patient-clustered bootstrap intervals.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. About 5 minutes.

Cell 4 checks the Brier score and calibration error against the published values (0.220 and 0.343) before the new quantities are reported.

In [ ]:
#@title 1. Environment (same pins as the original run)
!pip install -q xgboost==3.2.0 scikit-learn==1.8.0 imbalanced-learn==0.14.1 ucimlrepo 2>/dev/null
import numpy as np, pandas as pd, sklearn, xgboost as xgb, warnings, ssl, urllib.request
warnings.filterwarnings('ignore')
print('sklearn', sklearn.__version__, '| xgboost', xgb.__version__)
try:
    d = xgb.DMatrix(np.zeros((16,3)), label=np.array([0,1]*8))
    xgb.train({'tree_method':'hist','device':'cuda'}, d, num_boost_round=1); USE_GPU=True
except Exception: USE_GPU=False
print('GPU', USE_GPU)

In [ ]:
#@title 2. Build the baseline cohort from the public UCI dataset
ctx = ssl.create_default_context(); ctx.check_hostname=False; ctx.verify_mode=ssl.CERT_NONE
_o = urllib.request.urlopen
urllib.request.urlopen = lambda *a, **k: _o(*a, context=ctx, **{kk:vv for kk,vv in k.items() if kk!='context'})
from ucimlrepo import fetch_ucirepo
d = fetch_ucirepo(id=296)
raw = pd.concat([p for p in [d.data.ids, d.data.features, d.data.targets] if p is not None], axis=1)

EXPIRED={11,13,14,19,20,21}
DR={'has_diabetes_dx':[(250,250.99)],'has_circulatory_dx':[(390,459)],'has_respiratory_dx':[(460,519)],
    'has_renal_dx':[(580,629)],'has_digestive_dx':[(520,579)],'has_infectious_dx':[(1,139)],
    'has_injury_dx':[(800,999)],'has_neoplasm_dx':[(140,239)],'has_symptoms_dx':[(780,799)]}
u = raw.replace('?', np.nan).copy()
u = u[~u['discharge_disposition_id'].isin(EXPIRED)].copy()
for c in ['diag_1','diag_2','diag_3']:
    u[c]=pd.to_numeric(u[c].astype(str).str.replace('V|E','10',regex=True),errors='coerce')
for dis,rg in DR.items():
    m=False
    for lo,hi in rg: m = m | u[['diag_1','diag_2','diag_3']].apply(lambda col: col.between(lo,hi)).any(axis=1)
    u[dis]=m.astype(int)
MED=['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride','acetohexamide','glipizide',
 'glyburide','tolbutamide','pioglitazone','rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
 'examide','citoglipton','insulin','glyburide-metformin','glipizide-metformin','glimepiride-pioglitazone',
 'metformin-rosiglitazone','metformin-pioglitazone']
med=[c for c in MED if c in u.columns]
u['med_change_count']=u[med].isin(['Up','Down']).sum(axis=1)
u['comorbidity_count']=u[list(DR)].sum(axis=1)
u['total_prior_visits']=(u['number_inpatient'].fillna(0)+u['number_emergency'].fillna(0)+u['number_outpatient'].fillna(0))
u=u.drop(columns=['diag_1','diag_2','diag_3'])
amap={'[0-10)':None,'[10-20)':None,'[20-30)':'20-39','[30-40)':'20-39','[40-50)':'40-59','[50-60)':'40-59',
 '[60-70)':'>=60','[70-80)':'>=60','[80-90)':'>=60','[90-100)':'>=60'}
u['age']=u['age'].map(amap); u=u.dropna(subset=['age'])
u=u[u['gender'].isin(['Male','Female'])]
u['gender']=u['gender'].map({'Male':1,'Female':2}).astype(int)
u['race']=u['race'].map({'Caucasian':3,'AfricanAmerican':4,'Hispanic':2,'Asian':5,'Other':5}).fillna(5).astype(int)
u['readmitted']=(u['readmitted'].astype(str)=='<30').astype(int)
u=u.drop(columns=[c for c in ['encounter_id','weight','payer_code','medical_specialty'] if c in u.columns])
u['patient_nbr']=u['patient_nbr'].astype(int)
base=u.drop(columns=['on_insulin'],errors='ignore')
assert (len(base), base.patient_nbr.nunique(), int(base.readmitted.sum()))==(98490,69311,11271)
print('cohort matches:', len(base), base.patient_nbr.nunique(), int(base.readmitted.sum()))

In [ ]:
#@title 3. Out-of-fold predictions from the baseline model (factorial cell D)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

CAT=['admission_type_id','discharge_disposition_id','admission_source_id','race']
y=base['readmitted'].values; g=base['patient_nbr'].values
X=base.drop(columns=['readmitted','patient_nbr'])
for c in CAT:
    if c in X.columns: X[c]=X[c].astype(str)
cat=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
X=pd.get_dummies(X,columns=cat,dummy_na=False).astype(float)
print('encoded predictors:',X.shape[1],'(expect 155)')

pos_w=(y==0).sum()/max((y==1).sum(),1)
def mk(seed=42):
    kw=dict(n_estimators=500,max_depth=4,learning_rate=0.03,subsample=0.8,colsample_bytree=0.7,
            min_child_weight=5,reg_lambda=2.0,scale_pos_weight=pos_w,eval_metric='logloss',
            random_state=seed,tree_method='hist')
    if USE_GPU: kw['device']='cuda'
    return XGBClassifier(**kw)

Xv=X.values.astype(float)
oof=np.zeros(len(y))
for tr,te in StratifiedGroupKFold(5,shuffle=True,random_state=42).split(np.zeros(len(y)),y,g):
    sc=StandardScaler().fit(Xv[tr])
    m=mk(); m.fit(sc.transform(Xv[tr]),y[tr])
    oof[te]=m.predict_proba(sc.transform(Xv[te]))[:,1]
print('out-of-fold predictions ready')

In [ ]:
#@title 4. Sanity check against the published calibration numbers
from sklearn.metrics import roc_auc_score, brier_score_loss

def ece_mce(y,p,bins=10):
    edges=np.linspace(0,1,bins+1); e=0.0; m=0.0
    for i in range(bins):
        sel=(p>=edges[i])&(p<edges[i+1]) if i<bins-1 else (p>=edges[i])&(p<=edges[i+1])
        if sel.sum()==0: continue
        gap=abs(y[sel].mean()-p[sel].mean())
        e+=sel.mean()*gap; m=max(m,gap)
    return e,m

auc=roc_auc_score(y,oof); brier=brier_score_loss(y,oof); ece,mce=ece_mce(y,oof)
print(f'AUROC  {auc:.4f}   published 0.671')
print(f'Brier  {brier:.4f}   published 0.220')
print(f'ECE    {ece:.4f}   published 0.343')
print(f'MCE    {mce:.4f}   published 0.449')
print('\nIf these are close, the calibration measures below describe the same model.')

In [ ]:
#@title 5. Calibration slope and calibration-in-the-large, with patient-clustered bootstrap CIs
import statsmodels.api as sm

EPS=1e-9
def logit(p): p=np.clip(p,EPS,1-EPS); return np.log(p/(1-p))

def cal_slope_citl(y,p):
    """Cox calibration. Slope: logistic regression of y on logit(p).
       CITL: intercept of a logistic regression of y on logit(p) with the slope fixed at 1."""
    lp=logit(p)
    slope_fit=sm.GLM(y, sm.add_constant(lp), family=sm.families.Binomial()).fit()
    slope=slope_fit.params[1]; slope_int=slope_fit.params[0]
    citl_fit=sm.GLM(y, np.ones((len(y),1)), family=sm.families.Binomial(), offset=lp).fit()
    return slope, slope_int, citl_fit.params[0]

slope, slope_int, citl = cal_slope_citl(y,oof)
print(f'calibration slope        {slope:.4f}   (ideal 1)')
print(f'  intercept of that fit  {slope_int:.4f}')
print(f'calibration-in-the-large {citl:.4f}   (ideal 0)')
print(f'mean predicted risk      {oof.mean():.4f}   observed {y.mean():.4f}')

#@markdown Bootstrap resamples (patients, not encounters)
N_BOOT = 500  #@param {type:"integer"}
rng=np.random.default_rng(42)
pat=np.unique(g); idx_by_pat={pp:np.where(g==pp)[0] for pp in pat}
bs=[]
for b in range(N_BOOT):
    pick=rng.choice(pat,size=len(pat),replace=True)
    idx=np.concatenate([idx_by_pat[pp] for pp in pick])
    try: bs.append(cal_slope_citl(y[idx],oof[idx]))
    except Exception: pass
    if (b+1)%100==0: print(f'  {b+1}/{N_BOOT}',flush=True)
bs=np.array(bs)
sl_ci=np.percentile(bs[:,0],[2.5,97.5]); ci_ci=np.percentile(bs[:,2],[2.5,97.5])
print()
print(f'calibration slope        {slope:.3f} (95% CI {sl_ci[0]:.3f} to {sl_ci[1]:.3f})')
print(f'calibration-in-the-large {citl:.3f} (95% CI {ci_ci[0]:.3f} to {ci_ci[1]:.3f})')

In [ ]:
#@title 6. Same measures after within-training-fold recalibration
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

oof_platt=np.zeros(len(y)); oof_iso=np.zeros(len(y))
for tr,te in StratifiedGroupKFold(5,shuffle=True,random_state=42).split(np.zeros(len(y)),y,g):
    # patient-grouped 20% calibration slice inside the training fold only
    tr_pat=np.unique(g[tr]); rs=np.random.default_rng(42)
    cal_pat=set(rs.choice(tr_pat,size=int(0.2*len(tr_pat)),replace=False))
    is_cal=np.array([gg in cal_pat for gg in g[tr]])
    fit_idx=tr[~is_cal]; cal_idx=tr[is_cal]
    sc=StandardScaler().fit(Xv[fit_idx])
    m=mk(); m.fit(sc.transform(Xv[fit_idx]),y[fit_idx])
    p_cal=m.predict_proba(sc.transform(Xv[cal_idx]))[:,1]
    p_te =m.predict_proba(sc.transform(Xv[te]))[:,1]
    pl=LogisticRegression().fit(logit(p_cal).reshape(-1,1),y[cal_idx])
    oof_platt[te]=pl.predict_proba(logit(p_te).reshape(-1,1))[:,1]
    iso=IsotonicRegression(out_of_bounds='clip').fit(p_cal,y[cal_idx])
    oof_iso[te]=iso.predict(p_te)

for nm,pp in [('as reported',oof),('Platt',oof_platt),('isotonic',oof_iso)]:
    s,_,c=cal_slope_citl(y,pp); e,_=ece_mce(y,pp)
    print(f'{nm:12s} AUROC {roc_auc_score(y,pp):.4f}  Brier {brier_score_loss(y,pp):.4f}  '
          f'ECE {e:.4f}  slope {s:.3f}  CITL {c:+.3f}')

In [ ]:
#@title 7. Paste-ready sentence
s_rep,_,c_rep = cal_slope_citl(y,oof)
s_pl,_,c_pl   = cal_slope_citl(y,oof_platt)
txt = (f'The model as reported had a calibration slope of {s_rep:.2f} '
       f'(95% CI {sl_ci[0]:.2f} to {sl_ci[1]:.2f}) and calibration-in-the-large of {c_rep:+.2f} '
       f'(95% CI {ci_ci[0]:+.2f} to {ci_ci[1]:+.2f}), confirming that class weighting displaces '
       f'predicted probabilities upward relative to the 11.4% prevalence without destroying the '
       f'ordering. After within-training-fold Platt recalibration the slope was {s_pl:.2f} and '
       f'calibration-in-the-large {c_pl:+.2f}.')
print(txt)
open('calibration_slope.txt','w').write(txt)
from google.colab import files; files.download('calibration_slope.txt')